In [1]:
import sklearn
import scipy 
import numpy as np
import pandas as pd
import os
import sys

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "0"
sys.path.append(os.path.abspath("src"))
import jdcoot

sourceDomainName = ['amazon'] #['caltech10','amazon','webcam']
targetDomainName = ['amazon'] #['caltech10','amazon','webcam']

tests = []
data_source = {}
data_target = {}

min_max_scaler = sklearn.preprocessing.MinMaxScaler()
# Collab
possible_data = scipy.io.loadmat('caltech10_caffe.mat')
feat = possible_data['fts'].astype(float)
labels = possible_data['labels'].ravel()
S_data = [feat, labels]
S_nClass = len(np.unique(labels)) # nb de class in source data
possible_data = scipy.io.loadmat('caltech10_google.mat')
feat = possible_data['fts'].astype(float)
labels = possible_data['labels'].ravel()
T_data = [feat, labels]
T_nClass = len(np.unique(labels)) # nb de class in target data

source=pd.DataFrame(np.concatenate((S_data[0],S_data[1].reshape(-1,1)),axis=1),columns=['X'+str(i) for i in range(S_data[0].shape[1])]+['Z'])
target=pd.DataFrame(np.concatenate((T_data[0],T_data[1].reshape(-1,1)),axis=1),columns=['X'+str(i) for i in range(T_data[0].shape[1])]+['Z'])

source.loc[:,'Z'] = source.loc[:, 'Z'] - 1
target.loc[:,'Z'] = target.loc[:, 'Z'] - 1

E0000 00:00:1766671149.000880 2883090 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766671149.006032 2883090 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [13]:
def get_data(x, y, nbtrain, nbtest, nseed):
 
    y = y.ravel()
    n = x.shape[0]

    if nbtrain + nbtest > n:
        raise ValueError("nbtrain + nbtest exceeds total number of samples")

    np.random.seed(nseed)
    idx = np.random.permutation(n)

    train_idx = idx[:nbtrain]
    test_idx = idx[nbtrain:nbtrain + nbtest]

    xtrain = x[train_idx]
    ytrain = y[train_idx]

    xtest = x[test_idx]
    ytest = y[test_idx]

    return xtrain, ytrain, xtest, ytest


In [5]:
from jdcoot.models.discrete_unsupervised_jdcoot import discrete_unsupervised_jdcoot
from jdcoot.models.discrete_semisupervised_jdcoot import discrete_semisupervised_jdcoot
from jdcoot.models.discrete_partial_jdcoot import discrete_partial_jdcoot
from jdcoot.models.discrete_unsupervised_coot import discrete_unsupervised_coot
from jdcoot.models.discrete_semisupervised_coot import discrete_semisupervised_coot
from jdcoot.models.discrete_partial_coot import discrete_partial_coot
from jdcoot.models.discrete_semisupervised_reference import discrete_semisupervised_reference
from jdcoot.models.discrete_partial_reference import discrete_partial_reference



In [6]:
import numpy as np
nbtrain=80
nbtest=20
XtotS = np.array(S_data[0], dtype=float)
YtotS = np.array(S_data[1], dtype=int).reshape(-1, 1) - 1
XtotT = np.array(T_data[0], dtype=float)
YtotT = np.array(T_data[1], dtype=int).reshape(-1, 1) - 1
repe = 1

XS1,yS1,XStest,yStest = get_data(XtotS,YtotS,nbtrain,nbtest,repe)
XT1,yT1,XTtest,yTtest = get_data(XtotT,YtotT,nbtrain,nbtest,repe)

    #XS, yS = generateSubset(XS1, yS1, perClassSource)
    #XT, yT = generateSubset(XT1, yT1, perClassSource)
S = pd.DataFrame(np.c_[XS1, yS1],
                     columns=['X' + str(i) for i in range(XS1.shape[1])] + ['Z'])
T = pd.DataFrame(np.c_[XT1, yT1],
                     columns=['X' + str(i) for i in range(XT1.shape[1])] + ['Z'])
S_test = pd.DataFrame(np.c_[XStest, yStest],
                          columns=['X' + str(i) for i in range(XStest.shape[1])] + ['Z'])
T_test = pd.DataFrame(np.c_[XTtest, yTtest],
                          columns=['X' + str(i) for i in range(XTtest.shape[1])] + ['Z'])


# Exemple de valeurs à tester pour alpha
alpha_values = np.arange(0, 4.01, 0.5)   # 0, 0.1, 0.2, ..., 1.0
best_alpha = None
best_score = -np.inf  # ou 0 selon ta métrique

for a in alpha_values:

    pure_source, pure_target, test_source, test_target = discrete_unsupervised_jdcoot(source, target, source, target, alpha=a)

    score = test_target  
    
    if score > best_score:
        best_score = score
        best_alpha = a

print("Meilleur alpha :", best_alpha)
print("Score associé :", best_score)

[0 1 2 3 4 5 6 7 8 9]
[0 1 2 3 4 5 6 7 8 9]


I0000 00:00:1766671343.394051 2883090 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 470 MB memory:  -> device: 0, name: Tesla V100S-PCIE-32GB, pci bus id: 0000:3b:00.0, compute capability: 7.0
I0000 00:00:1766671345.275930 2883470 service.cc:148] XLA service 0x70d5fdc852d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1766671345.275968 2883470 service.cc:156]   StreamExecutor device (0): Tesla V100S-PCIE-32GB, Compute Capability 7.0
I0000 00:00:1766671345.293814 2883470 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1766671345.399102 2883470 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Delta: 0.020255819472126384 	  Loss: 2.199211259965066 	 Accuracy: 0.13446126447016918
Delta: 0.014972405370403894 	  Loss: 2.1989955211938463 	 Accuracy: 0.13446126447016918
Delta: 0.016612462978308498 	  Loss: 2.1988846700056266 	 Accuracy: 0.13446126447016918
Delta: 0.018277356241225073 	  Loss: 2.19829457870092 	 Accuracy: 0.13535173642030277
Delta: 0.016763675094514454 	  Loss: 2.197712264507718 	 Accuracy: 0.13446126447016918
Delta: 0.015597660987935351 	  Loss: 2.196560733370316 	 Accuracy: 0.12644701691896706
Delta: 0.0147186626428934 	  Loss: 2.194574724254459 	 Accuracy: 0.12288512911843277
Delta: 0.01476511939470042 	  Loss: 2.1915773402157965 	 Accuracy: 0.1246660730186999
Delta: 0.014139803653254214 	  Loss: 2.188527905220483 	 Accuracy: 0.12644701691896706
Delta: 0.014880333420051975 	  Loss: 2.184547622187754 	 Accuracy: 0.12288512911843277
Delta: 0.02046712018303437 	  Loss: 2.1711045277850944 	 Accuracy: 0.26714158504007124
Delta: 0.017466784970962126 	  Loss: 2.118946

/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.006826229221174854 	  Loss: 1.6394072717302222 	 Accuracy: 0.4479073909171861
Delta: 0.021139936869476135 	  Loss: 2.137720937774647 	 Accuracy: 0.26714158504007124
Delta: 0.018736876034559705 	  Loss: 2.018295786905181 	 Accuracy: 0.3241317898486198
Delta: 0.017710603639086248 	  Loss: 1.894115937623849 	 Accuracy: 0.3187889581478183
Delta: 0.016580472440510082 	  Loss: 1.7732711848495075 	 Accuracy: 0.26981300089047194
Delta: 0.013856616388424252 	  Loss: 1.7025737753070458 	 Accuracy: 0.27960819234194123
Delta: 0.011297688200365173 	  Loss: 1.667787588736679 	 Accuracy: 0.28138913624220835
Delta: 0.008083394717144893 	  Loss: 1.651907154182398 	 Accuracy: 0.2965271593944791


...

Delta: 0.007638857545539232 	  Loss: 1.6257685200016572 	 Accuracy: 0.37310774710596617
Delta: 0.005525722632387978 	  Loss: 1.6241537593097464 	 Accuracy: 0.37043633125556547
Delta: 0.00374940237771424 	  Loss: 1.6239403880957015 	 Accuracy: 0.3686553873552983
Delta: 0.003058994887425734 	  Loss: 1.6237648146402972 	 Accuracy: 0.371326803205699
Delta: 0.0220973439479129 	  Loss: 1.9964673159307165 	 Accuracy: 0.30365093499554763
Delta: 0.019815837289312254 	  Loss: 1.7949485878888929 	 Accuracy: 0.3374888691006233
Delta: 0.017897947391871956 	  Loss: 1.694430712256594 	 Accuracy: 0.3472840605520926
Delta: 0.016110216971803655 	  Loss: 1.6281937729768403 	 Accuracy: 0.3757791629563669
Delta: 0.012987572504798657 	  Loss: 1.5932155022413415 	 Accuracy: 0.39893143365983974


/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.010111762506248602 	  Loss: 1.5770684661027898 	 Accuracy: 0.41317898486197685
Delta: 0.007587831116163659 	  Loss: 1.5700147212944815 	 Accuracy: 0.41585040071237755
Delta: 0.006114985820372909 	  Loss: 1.567513164001266 	 Accuracy: 0.42030276046304543
Delta: 0.0034558219764672824 	  Loss: 1.5668947295298303 	 Accuracy: 0.41763134461264473
Delta: 0.0034600777730345355 	  Loss: 1.5668601274436522 	 Accuracy: 0.42119323241317896
Delta: 0.022209662152709617 	  Loss: 1.9783352838166608 	 Accuracy: 0.2983081032947462
Delta: 0.01981198219667149 	  Loss: 1.775316320730719 	 Accuracy: 0.352626892252894
Delta: 0.017857247557752275 	  Loss: 1.6787291043054673 	 Accuracy: 0.3196794300979519
Delta: 0.015436910081205941 	  Loss: 1.6265487505012168 	 Accuracy: 0.3383793410507569
Delta: 0.012759139109492204 	  Loss: 1.6012059299300674 	 Accuracy: 0.35529830810329477
Delta: 0.011393700648149865 	  Loss: 1.5851322571790973 	 Accuracy: 0.3686553873552983


/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.010151245910680881 	  Loss: 1.5745233338441849 	 Accuracy: 0.3722172751558326
Delta: 0.007808550765554239 	  Loss: 1.5698424576119827 	 Accuracy: 0.3748886910062333
Delta: 0.005566732322717676 	  Loss: 1.5682890961790332 	 Accuracy: 0.3757791629563669
Delta: 0.0040376089322940135 	  Loss: 1.5679172377800326 	 Accuracy: 0.377560106856634
Meilleur alpha : 0.5
Score associé : 0.4479073909171861


In [14]:
results = []
numRepetitions = 10
nbtrain = 800
nbtest = 200
a = 0.4

prop_target_values_s = [0.01, 0.05, 0.1, 0.2, 0.4]
prop_target_values_p = [0.01, 0.05, 0.1, 0.2, 0.4]

XtotS = np.array(S_data[0], dtype=float)
YtotS = np.array(S_data[1], dtype=int).reshape(-1, 1) - 1

XtotT = np.array(T_data[0], dtype=float)
YtotT = np.array(T_data[1], dtype=int).reshape(-1, 1) - 1

for repe in range(numRepetitions):
    print("num repe :", repe + 1)

    XS1, yS1, XStest, yStest = get_data(XtotS, YtotS, nbtrain, nbtest, repe)
    XT1, yT1, XTtest, yTtest = get_data(XtotT, YtotT, nbtrain, nbtest, repe)

    S = pd.DataFrame(
        np.c_[XS1, yS1],
        columns=[f"X{i}" for i in range(XS1.shape[1])] + ["Z"]
    )

    T = pd.DataFrame(
        np.c_[XT1, yT1],
        columns=[f"X{i}" for i in range(XT1.shape[1])] + ["Z"]
    )

    S_test = pd.DataFrame(
        np.c_[XStest, yStest],
        columns=[f"X{i}" for i in range(XStest.shape[1])] + ["Z"]
    )

    T_test = pd.DataFrame(
        np.c_[XTtest, yTtest],
        columns=[f"X{i}" for i in range(XTtest.shape[1])] + ["Z"]
    )

    # =========================================================
    # UNSUPERVISED
    # =========================================================
     # COOT
    pure_source, pure_target, test_source, test_target = \
        discrete_unsupervised_coot(S, T, S_test, T_test)

    results.append({
        "repetition": repe,
        "recoding": "coot",
        "learning": "unsupervised",
        "prop_source": 1,
        "prop_target": 0,
        "pure_source": pure_source,
        "test_source": test_source,
        "pure_target": pure_target,
        "test_target": test_target,
    })
    
    # JDCOOT
    pure_source, pure_target, test_source, test_target = \
        discrete_unsupervised_jdcoot(S, T, S_test, T_test, alpha=a)

    results.append({
        "repetition": repe,
        "recoding": "jdcoot",
        "learning": "unsupervised",
        "prop_source": 1,
        "prop_target": 0,
        "pure_source": pure_source,
        "test_source": test_source,
        "pure_target": pure_target,
        "test_target": test_target,
    })

   
import pandas as pd

df_results = pd.DataFrame(results)


df_results

df_summary = (
    df_results
    .groupby(
        ["recoding", "learning", "prop_source", "prop_target"],
        as_index=False
    )
    .agg(
        pure_source_mean=("pure_source", "mean"),
        pure_source_var=("pure_source", "var"),
        test_source_mean=("test_source", "mean"),
        test_source_var=("test_source", "var"),
        pure_target_mean=("pure_target", "mean"),
        pure_target_var=("pure_target", "var"),
        test_target_mean=("test_target", "mean"),
        test_target_var=("test_target", "var"),
    )
)

df_summary
  

num repe : 1
Delta:       0.0156359 	 Loss:       2.2001753


Delta:       0.0208151 	 Loss:       2.1664660
Delta:       0.0177443 	 Loss:       2.1378616
Delta:       0.0151617 	 Loss:       2.1216158
Delta:       0.0132620 	 Loss:       2.1119749
Delta:       0.0116932 	 Loss:       2.1051573
Delta:       0.0090593 	 Loss:       2.1014896
Delta:       0.0063191 	 Loss:       2.1005522
Delta:       0.0048636 	 Loss:       2.1002913
Delta:       0.0031599 	 Loss:       2.1001825
Delta:       0.0031378 	 Loss:       2.1002385
Delta:       0.0032486 	 Loss:       2.1002115
Delta:       0.0016972 	 Loss:       2.1001982
Delta:       0.0021360 	 Loss:       2.1001601
Delta:       0.0014302 	 Loss:       2.1001308
Delta:       0.0010391 	 Loss:       2.1001257
Delta:       0.0000011 	 Loss:       2.1001270
Delta:       0.0000000 	 Loss:       2.1001270
converged at iter  17
Delta: 0.020565340734303172 	  Loss: 2.179369831981312 	 Accuracy: 0.2475
Delta: 0.017793093734844667 	  Loss: 2.1362575883056776 	 Accuracy: 0.2675
Delta: 0.01691620669720565 	  

/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.008673582956440289 	  Loss: 1.743111702832529 	 Accuracy: 0.38125
num repe : 3
Delta:       0.0156356 	 Loss:       2.1923470
Delta:       0.0207089 	 Loss:       2.1577418
Delta:       0.0176410 	 Loss:       2.1315526
Delta:       0.0147688 	 Loss:       2.1184375
Delta:       0.0121449 	 Loss:       2.1107606
Delta:       0.0106553 	 Loss:       2.1056453
Delta:       0.0087416 	 Loss:       2.1032827
Delta:       0.0086679 	 Loss:       2.1020481
Delta:       0.0090215 	 Loss:       2.1006614
Delta:       0.0097421 	 Loss:       2.0981950
Delta:       0.0086315 	 Loss:       2.0954841
Delta:       0.0070036 	 Loss:       2.0940530
Delta:       0.0049179 	 Loss:       2.0933799
Delta:       0.0033443 	 Loss:       2.0930857
Delta:       0.0037750 	 Loss:       2.0930008
Delta:       0.0015891 	 Loss:       2.0928823
Delta:       0.0000005 	 Loss:       2.0928823
converged at iter  16
Delta: 0.02049267782605962 	  Loss: 2.1700797414482245 	 Accuracy: 0.23375
Delta: 0.0184518

/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


...

/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.006721993869176708 	  Loss: 1.6236493389606959 	 Accuracy: 0.5675
num repe : 8
Delta:       0.0156361 	 Loss:       2.1946524
Delta:       0.0206580 	 Loss:       2.1611749
Delta:       0.0181739 	 Loss:       2.1300223
Delta:       0.0154619 	 Loss:       2.1003057
Delta:       0.0127411 	 Loss:       2.0870107
Delta:       0.0099782 	 Loss:       2.0821693
Delta:       0.0073979 	 Loss:       2.0802177
Delta:       0.0051541 	 Loss:       2.0797177
Delta:       0.0041000 	 Loss:       2.0795023
Delta:       0.0009220 	 Loss:       2.0794279
Delta:       0.0000011 	 Loss:       2.0794264
Delta:       0.0000000 	 Loss:       2.0794264
converged at iter  11
Delta: 0.02054709504447819 	  Loss: 2.1731389591884422 	 Accuracy: 0.2525
Delta: 0.018293940348774296 	  Loss: 2.121391941053391 	 Accuracy: 0.29875
Delta: 0.016477123391664964 	  Loss: 2.0237526111005 	 Accuracy: 0.32625
Delta: 0.015234479061259305 	  Loss: 1.9063114269336152 	 Accuracy: 0.2925
Delta: 0.013788258327846711 	

/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.008400603125010868 	  Loss: 1.6725630766525375 	 Accuracy: 0.22625
num repe : 9
Delta:       0.0156354 	 Loss:       2.1909716
Delta:       0.0206670 	 Loss:       2.1556383
Delta:       0.0176327 	 Loss:       2.1259861
Delta:       0.0153011 	 Loss:       2.1063293
Delta:       0.0136336 	 Loss:       2.0947771
Delta:       0.0134920 	 Loss:       2.0848721
Delta:       0.0121640 	 Loss:       2.0771261
Delta:       0.0110839 	 Loss:       2.0722470
Delta:       0.0094331 	 Loss:       2.0693896
Delta:       0.0073960 	 Loss:       2.0681016
Delta:       0.0054370 	 Loss:       2.0676296
Delta:       0.0039224 	 Loss:       2.0674446
Delta:       0.0028799 	 Loss:       2.0672064
Delta:       0.0024246 	 Loss:       2.0671464
Delta:       0.0021062 	 Loss:       2.0671286
Delta:       0.0009799 	 Loss:       2.0671061
Delta:       0.0000018 	 Loss:       2.0671048
Delta:       0.0000000 	 Loss:       2.0671048
converged at iter  17
Delta: 0.020486253526311355 	  Loss: 2.1687

,recoding,learning,prop_source,prop_target,pure_source_mean,pure_source_var,test_source_mean,test_source_var,pure_target_mean,pure_target_var,test_target_mean,test_target_var
0,coot,unsupervised,1,0,1.0,0.0,1.0,0.0,0.382625,0.029802,0.3785,0.028606
1,jdcoot,unsupervised,1,0,1.0,0.0,1.0,0.0,0.383875,0.015073,0.3970,0.016362


In [16]:
df_results

,repetition,recoding,learning,prop_source,prop_target,pure_source,test_source,pure_target,test_target
0,0,coot,unsupervised,1,0,1.0,1.0,0.20125,0.200
1,0,jdcoot,unsupervised,1,0,1.0,1.0,0.44000,0.480
2,1,coot,unsupervised,1,0,1.0,1.0,0.72875,0.695
3,1,jdcoot,unsupervised,1,0,1.0,1.0,0.38125,0.355
4,2,coot,unsupervised,1,0,1.0,1.0,0.30250,0.345
5,2,jdcoot,unsupervised,1,0,1.0,1.0,0.38125,0.365
6,3,coot,unsupervised,1,0,1.0,1.0,0.18625,0.175
7,3,jdcoot,unsupervised,1,0,1.0,1.0,0.54000,0.535
8,4,coot,unsupervised,1,0,1.0,1.0,0.54875,0.545
9,4,jdcoot,unsupervised,1,0,1.0,1.0,0.35500,0.400
